# Week 2, Lab 1: String Surgery - Teaching Your Digital Twin to Read

## The map before the territory

In week 1 your digital twin learned to talk. This week it learns to **remember things it was never trained on**, like your CV, your notes or your bio. That whole trick has four steps, one per lab:

1. **Read** (this lab): cut a long document into small, clean pieces called *chunks*.
2. **Manage the working memory** (lab 2): fit the right text into the model's limited space.
3. **Turn text into numbers that capture meaning** (lab 3): these are called *embeddings*.
4. **Retrieve and answer** (lab 4): find the best chunks and hand them to the model. This is called *RAG*.

Everything starts with plain strings. A model cannot swallow a whole book at once, so your code has to cut the text up first. This lab builds that cutting tool. There is no API call. The Reading tab explains slicing, `split` and `join`, and stepping through a list from scratch if you want a refresher first.

## Exercise

### Goal
Write a small tool that cuts a long text into *chunks*. A *chunk* is a short piece of the text, here a fixed number of words. Each chunk also repeats a few words from the end of the one before it. That repeat is called *overlap*. Overlap matters because a fact can sit right on the cut. Without overlap, half of it would end up in one chunk and half in the next, and neither chunk would make sense alone.

### 1. `chunk_words(text, size, overlap)`
Code shape (replace each `...`):
```python
def chunk_words(text, size, overlap):
    if ...:
        raise ValueError(...)
    words = text.split()
    step = ...
    chunks = []
    for start in range(0, len(words), step):
        chunks.append(...)
        if ...:
            break
    return chunks
```
**You get:** `text` (a string), `size` (how many words go in one chunk) and `overlap` (how many words one chunk shares with the next).
**You give back:** a list of strings. Each string is one chunk, with its words joined by single spaces.
**If something is wrong:** if `overlap` is not smaller than `size`, raise a `ValueError`. `ValueError` is the standard Python error for "you gave me a bad value". You make one with `raise ValueError("message")`. The reason: if the overlap is as big as the chunk, you would never move forward.

**Idea:**
1. Check `overlap < size`. If not, raise the `ValueError`.
2. Cut the text into a list of words with `text.split()`. With no argument, `split` cuts at every run of spaces or new lines.
3. Work out how far to move for each new chunk: `step = size - overlap`. With `size=4` and `overlap=1` the step is 3.
4. Loop over the start positions with `range(0, len(words), step)`. This counts 0, 3, 6 and so on. For each `start`, take the slice `words[start:start + size]` and glue it into one string with `" ".join(...)`.
5. Stop after the chunk that reaches the end of the text. If `start + size >= len(words)`, `break`. Without this the loop would add a last chunk made only of words you already used.

- A text with fewer words than `size` gives one chunk with all the words.
- An empty text gives an empty list.
- Extra spaces and new lines in the text do not matter. `split()` removes them.

**Check (by hand):** `chunk_words("one two three four five six seven eight nine ten", 4, 1)` gives `["one two three four", "four five six seven", "seven eight nine ten"]`. Each chunk starts with the last word of the one before it.

### What the asserts below test
The cell below the function checks four things:
- the ten-word example above gives exactly those three chunks;
- a short text, `"a b c"`, with `size=5` gives one chunk, `["a b c"]`;
- an empty text gives `[]`;
- `overlap >= size` raises a `ValueError`. If your code gives something back, or raises a different error, the check fails.

If all four hold, you see `All checks passed.`.

In [ ]:
def chunk_words(text: str, size: int, overlap: int) -> list[str]:
    """Cut text into chunks of `size` words that overlap by `overlap` words.

    You get: a string, the chunk size in words, and the overlap in words.
    You give back: a list of strings, one per chunk, words joined by single spaces.
    It raises a ValueError if overlap is not smaller than size.
    """
    # TODO:
    # 1. If overlap >= size, raise a ValueError.
    # 2. words = text.split()
    # 3. step = size - overlap
    # 4. For each start in range(0, len(words), step), add " ".join(words[start:start + size]).
    # 5. After adding a chunk, break if start + size >= len(words) (that chunk reached the end).
    raise NotImplementedError("chunk_words() is not implemented yet - replace the TODO with your code")


doc = "one two three four five six seven eight nine ten"
expected = ["one two three four", "four five six seven", "seven eight nine ten"]
assert chunk_words(doc, 4, 1) == expected, "chunk_words(doc, 4, 1) should be " + repr(expected) + ", got " + repr(chunk_words(doc, 4, 1))
assert chunk_words("a b c", 5, 0) == ["a b c"], 'chunk_words("a b c", 5, 0) should be ["a b c"], got ' + repr(chunk_words("a b c", 5, 0))
assert chunk_words("", 4, 1) == [], 'chunk_words("", 4, 1) should be [], got ' + repr(chunk_words("", 4, 1))

try:
    chunk_words(doc, 3, 3)
    raise AssertionError("should have raised ValueError when overlap is not smaller than size")
except ValueError:
    pass

print("All checks passed.")


## Solution

In [ ]:
def chunk_words(text: str, size: int, overlap: int) -> list[str]:
    if overlap >= size:
        raise ValueError("overlap must be smaller than size")
    words = text.split()
    step = size - overlap
    chunks = []
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start:start + size]))
        if start + size >= len(words):
            break
    return chunks


print(chunk_words("one two three four five six seven eight nine ten", 4, 1))


## Real-world project (offline)

The code cell below loads the twin's document: a few lines about Ada, with messy spacing and blank lines, just like text copied from a real file. Your project cleans it up and cuts it into overlapping chunks with your `chunk_words`. Nothing here calls an LLM. These chunks are what the next labs will search through.

In [ ]:
"""Load the twin's document (embedded so this notebook needs no repo checkout)."""
# lab-data-loader
TWIN_DOC = """Ada writes Python code and builds web apps every day.
She bakes bread and cooks spicy soup on weekends.

On Saturdays she runs five kilometers in the park.   Last summer she travelled to Accra and Lisbon by plane.

She teaches evening classes about data and machine learning.
Her favorite team is Hearts of Oak and she watches every match."""

print(f"Loaded the twin's document: {len(TWIN_DOC.split())} words.")


In [ ]:
clean = " ".join(TWIN_DOC.split())
chunks = chunk_words(clean, 12, 4)

print(f"{len(clean.split())} words -> {len(chunks)} chunks")
for number, chunk in enumerate(chunks, 1):
    print(f"[{number}] {chunk}")
